# PROBLEMAS

In [74]:
from shapely.validation import explain_validity
invalid_OD_zones_gdf = OD_zones_gdf_untreated[~OD_zones_gdf_untreated.is_valid]
explain_validity(invalid_OD_zones_gdf)

# Por muitos polígonos não são válidos, (declarado na célula 3) segue a lista de razões:

,geometry
8,Ring Self-intersection[334100.113609622 739806...
10,Self-intersection[335766.610853515 7397060.310...
11,Ring Self-intersection[335762.685647727 739705...
39,Ring Self-intersection[338658.671142381 739572...
40,Ring Self-intersection[338658.704127304 739572...
...,...
511,Ring Self-intersection[315652.949310819 740127...
512,Ring Self-intersection[304370.332846271 739657...
513,Ring Self-intersection[304370.332846271 739657...
517,Self-intersection[304438.957977715 7382590.458...


# CEMITÉRIO

In [27]:
import pandas as pd

def read_ibge_microdata(txt_path, layout_path, sheet_name='PESS'):
    # Read layout sheet
    layout = pd.read_excel(layout_path, sheet_name=sheet_name, engine='odf').iloc[1:]
    
    # Extract variable names and positions
    var_names = layout.iloc[:, 0].astype(str).tolist()
    starts = layout.iloc[:, 7].astype(float).astype(int).tolist()
    ends = layout.iloc[:, 8].astype(float).astype(int).tolist()
    
    # Build colspecs
    colspecs = [(start - 1, end) for start, end in zip(starts, ends)]
    
    # Read fixed-width file
    df = pd.read_fwf(txt_path, colspecs=colspecs, names=var_names)
    
    return df

txt_path = '.\\data\\Amostra_Pessoas_14munic.txt'
layout_path = '.\\Layout_microdados_Amostra.ods'

df = read_ibge_microdata(txt_path, layout_path)


In [1]:
dict_code_prof = {
    'V0601':{'sexo':{'1':'Masculino','2':'Feminino'}},
    'V0606':{'cor':{'1':'Branca','2':'Preta','3':'Amarela','4':'Parda','5':'Indigena','9':'Sem_declaracao'}},
    'V6400':{'nivel_de_instrucao':{'1':'Sem_instrucao','2':'fundamental_completo','3':'medio_completo','4':'superior_completo'}},
    'V0640':{'estado_civil':{'1':'casado','5':'solteiro','4':'viuvo','3':'divorciado','2':'desquitado'}},
    'V0645':{'trabalho':{'1':'um','2':'dois_ou_mais','':'nenhum'}},
    'V6526':'renda_em_salarios_minimos',
    'V0653':'horas_trabalhadas',
    'V0662':{'tempo_de_deslocamento_ao_trabalho':{'1':'<=5min','2':'6-30min','3':'31min-1h','4':'1-2h','5':'>2h'}},
    'V6940':{'categoria_profissional':{'1':'empregado_clt','2':'empregado_estatuario(militares_inclusos)','3':'empregado_sem_clt','4':'conta_propria','5':'empregador','6':'nao_remunerado','7':'trabalhador_subsistente'}},
    'V5080':'rendimento_familiar_per_capita_em_salarios_minimos',
    'V1005':{'situacao_do_setor':{'1':'area_urbanizada','2':'area_nao_urbanizada','3':'area_urbanizada_isolada','4':'area_rural_de_extensao_urbana','5':'aglomerado_rural','6':'aglomerado_rural','7':'aglomerado_rural','8':'area_rural_exclusive_aglomerado_rural'}},
    'V0221':{'motocicleta_para_uso_particular':{'1':'possui','2':'nao_possui'}},
    'V0222':{'automovel_para_uso_particular':{'1':'possui','2':'nao_possui'}},
}

# Separando os ciclistas gerais dos ciclistas vivendo na zona PNB

In [2]:
import geopandas as gpd
from shapely import union_all 
from shapely import make_valid 

OD_data_df = gpd.read_file('./data/od23_all.csv')

OD_data_gdf = gpd.GeoDataFrame(
    OD_data_df, 
    geometry=gpd.points_from_xy(OD_data_df['CO_DOM_Y'], OD_data_df['CO_DOM_X']),
    crs="EPSG:4326"
)
OD_data_gdf = OD_data_gdf.to_crs(epsg=31983)

OD_zones_gdf_untreated = gpd.read_file('./data/Zonas_2023.shp')
OD_zones_gdf = make_valid(OD_zones_gdf_untreated)
OD_zones_gdf.set_crs(epsg=31983, inplace=True)

spsp_limits = gpd.read_file('./data/REGIAO5/SIRGAS_REGIAO5.shp')

OD_zones_spsp = OD_zones_gdf.clip(spsp_limits.union_all())

spsp_data = OD_data_gdf[OD_data_gdf.geometry.within(OD_zones_spsp.union_all())]

spsp_cyclist_data = spsp_data['MODOPRIN'] == '16'

spsp_OD_data_gdf = spsp_data[spsp_cyclist_data]
# this dataframe contain the data of all cyclists living in Sao Paulo city
# in this particular dataset, those are all cyclists.

pnb_zone_gdf = gpd.read_file('./data/pnb_zone.shp')
pnb_zone = pnb_zone_gdf['geometry'][0]

spsp_pnb_OD = spsp_OD_data_gdf.geometry.apply(lambda p: pnb_zone.contains(p))
# long nonsense abreviation, sorry. It means the cyclists living in Sao Paulo's pnb zone

spsp_pnb_OD_data_gdf = spsp_OD_data_gdf[spsp_pnb_OD]
# this dataframe contains the cyclists living in the PNB zone of Sao Paulo city


### Teste: Há 'ciclistas secundários'?

In [3]:
secondary_cyclists = spsp_pnb_OD_data_gdf[(spsp_pnb_OD_data_gdf['MODOPRIN'].astype('int') != '16') &
                                          ((spsp_pnb_OD_data_gdf['MODO1'] == '16') |
                                           (spsp_pnb_OD_data_gdf['MODO2'] == '16') |
                                           (spsp_pnb_OD_data_gdf['MODO3'] == '16') |
                                           (spsp_pnb_OD_data_gdf['MODO4'] == '16')                                           
                                           )]
secondary_cyclists[['MODOPRIN', 'MODO1', 'MODO2', 'MODO3', 'MODO4']]
# why is row zero present? 

,MODOPRIN,MODO1,MODO2,MODO3,MODO4
0,16,16,0,0,0
1,16,16,0,0,0
93,16,16,0,0,0
94,16,16,0,0,0
371,16,16,0,0,0
...,...,...,...,...,...
75641,16,16,0,0,0
75835,16,16,0,0,0
75836,16,16,0,0,0
75858,16,16,0,0,0


## Em geral, não

In [ ]:
cyclist_profile = spsp_OD_data_gdf[['FE_PESS', 
                                    'ZONA',
                                    'SEXO', 
                                    'IDADE', 
                                    'RAÇA', 
                                    'RENDA_FA', 
                                    'GRAU_INS', 
                                    'TIPO_DOM',
                                    'DURACAO',
                                    'DISTANCIA',
                                    'PE_BICI',
                                    'MOT_SRES',
                                    'QT_AUTO',
                                    'QT_MOTO'
                                    ]]

cyclist_profile = cyclist_profile.astype({
    'FE_PESS': 'float',
    'RENDA_FA': 'float',
    'DURACAO': 'int',
    'DISTANCIA': 'float',
    'QT_AUTO': 'int',
    'QT_MOTO': 'int'})

pnb_cyclist_profile = spsp_pnb_OD_data_gdf[['FE_PESS',
                                            'ZONA',
                                            'SEXO', 
                                            'IDADE', 
                                            'RAÇA', 
                                            'RENDA_FA', 
                                            'GRAU_INS', 
                                            'TIPO_DOM',
                                            'DURACAO',
                                            'DISTANCIA',
                                            'PE_BICI',
                                            'MOT_SRES',
                                            'QT_AUTO',
                                            'QT_MOTO'
                                            ]]

pnb_cyclist_profile = pnb_cyclist_profile.astype({
    'FE_PESS': 'float',
    'RENDA_FA': 'float',
    'DURACAO': 'int',
    'DISTANCIA': 'float',
    'QT_AUTO': 'int',
    'QT_MOTO': 'int'})
    
def get_value(df):
    for col in df.columns:
        field = df[col]
        if type(field) == str:
            display(df[col].groupby([col]).count())



In [24]:
# get_value(cyclist_profile)
jorge = cyclist_profile[['IDADE']]
jorge


,IDADE
0,67
1,67
93,28
94,28
350,34
...,...
75641,73
75835,49
75836,49
75858,13
